# Sarcina 2 — Integrarea LLM & Logica de Dialog

**Scop:** Acest notebook implementează motorul de generare a răspunsurilor pentru un chatbot de suport pe social media.

**Flux:**
```
JSON (intent + confidence + entities)  ←  Sarcina 1
         ↓
build_system_prompt()    ← Prompt Engineering
         ↓
chat()                   ← Gestionare context + apel Groq API
         ↓
post_process()           ← Filtrare și validare răspuns
         ↓
generate_response()      ← Funcția finală expusă către Sarcina 3
```

**Model folosit:** `llama-3.3-70b-versatile` via Groq API  
**Dependențe:** `groq`, `google-colab`

## Pasul 1 — Instalare dependențe

In [ ]:
!pip install groq -q

## Pasul 2 — Configurare API Key

> ⚠️ **Important:** Nu scrie niciodată API key-ul direct în cod.  
> Mergi la **🔑 Secrets** (panoul stâng din Colab) → Add new secret  
> **Name:** `GROQ_API_KEY` | **Value:** *key-ul tău Groq*

In [ ]:
from google.colab import userdata
from groq import Groq

# Citim key-ul din Secrets — nu apare niciodată în cod
api_key = userdata.get('GROQ_API_KEY')

# Inițializăm clientul Groq
client = Groq(api_key=api_key)

# Modelul folosit
MODEL = "llama-3.3-70b-versatile"

print("✅ Client Groq inițializat cu succes")

## Pasul 3 — Mock Sarcina 1

Până când Sarcina 1 este gata, simulăm output-ul lor.  
**Contractul agreat:** funcția returnează mereu același format JSON.

```json
{
  "intent": "complaint",
  "confidence": 0.87,
  "entities": { "platform": "Instagram", "issue": "cont blocat" }
}
```

In [ ]:
def get_intent_mock(text: str) -> dict:
    """
    Mock pentru Sarcina 1.
    Înlocuiește această funcție cu apelul real când Sarcina 1 este gata.
    Input:  text (string) — mesajul utilizatorului
    Output: dict cu intent, confidence, entities
    """
    text_lower = text.lower()

    if any(w in text_lower for w in ["bună", "salut", "hello", "hey", "buna"]):
        return {"intent": "greeting", "confidence": 0.95, "entities": {}}

    elif any(w in text_lower for w in ["problemă", "nu merge", "eroare", "blocat", "problema"]):
        return {"intent": "complaint", "confidence": 0.88, "entities": {"issue": "eroare aplicație"}}

    elif any(w in text_lower for w in ["cum", "ce", "unde", "când", "?"]):
        return {"intent": "question", "confidence": 0.82, "entities": {}}

    elif any(w in text_lower for w in ["mulțumesc", "mersi", "super", "perfect"]):
        return {"intent": "feedback_positive", "confidence": 0.91, "entities": {}}

    elif any(w in text_lower for w in ["pa", "la revedere", "bye"]):
        return {"intent": "farewell", "confidence": 0.93, "entities": {}}

    else:
        return {"intent": "unknown", "confidence": 0.40, "entities": {}}


# Test rapid
print(get_intent_mock("Bună ziua!"))
print(get_intent_mock("Nu îmi merge aplicația"))
print(get_intent_mock("Cum îmi resetez parola?"))

## Pasul 4 — Prompt Engineering

Construim system prompt-ul dinamic în funcție de intentul primit de la Sarcina 1.  
Dacă `confidence < 0.5`, ignorăm intentul și tratăm mesajul generic.

In [ ]:
# Pragul minim de încredere pentru a folosi intentul
CONFIDENCE_THRESHOLD = 0.5
MAX_RESPONSE_LENGTH = 500

# Instrucțiuni specifice per intent
INTENT_HINTS = {
    "complaint":        "Utilizatorul are o problemă sau reclamație. Fii empatic, recunoaște problema și oferă o soluție concretă.",
    "question":         "Utilizatorul pune o întrebare. Răspunde direct, clar și util.",
    "greeting":         "Utilizatorul salută. Răspunde prietenos și întreabă cu ce îl poți ajuta.",
    "feedback_positive":"Utilizatorul este mulțumit. Mulțumește-i și închide conversația pozitiv.",
    "farewell":         "Utilizatorul își ia rămas bun. Răspunde scurt și politicos.",
    "unknown":          "Mesajul nu este clar. Cere politicos clarificări fără să fii confuzant."
}


def build_system_prompt(intent_data: dict) -> str:
    """
    Construiește system prompt-ul pentru LLM pe baza intentului detectat.
    Input:  intent_data (dict) — output-ul de la Sarcina 1
    Output: string — system prompt complet
    """
    intent = intent_data.get("intent", "unknown")
    confidence = intent_data.get("confidence", 0.0)
    entities = intent_data.get("entities", {})

    # Prompt de bază — comportamentul general al chatbot-ului
    base = """Ești un asistent de suport pentru o platformă de social media.
Răspunzi întotdeauna în limba română, scurt (maxim 3 propoziții), clar și prietenos.
Nu inventa informații. Dacă nu știi răspunsul, spune că vei escalada problema."""

    # Adaugă hint de intent doar dacă modelul e suficient de sigur
    if confidence >= CONFIDENCE_THRESHOLD and intent in INTENT_HINTS:
        base += f"\n\nContextul mesajului: {INTENT_HINTS[intent]}"

    # Injectează entitățile extrase dacă există
    if entities:
        entities_text = ", ".join(f"{k}: {v}" for k, v in entities.items())
        base += f"\n\nInformații detectate din mesaj: {entities_text}."

    return base


# Test
mock_intent = {"intent": "complaint", "confidence": 0.88, "entities": {"issue": "cont blocat"}}
print(build_system_prompt(mock_intent))

## Pasul 5 — Gestionarea Contextului Conversației

Chatbot-ul trebuie să "țină minte" ce s-a discutat anterior.  
Păstrăm un istoric per `user_id` și îl trimitem complet la fiecare apel LLM.

In [ ]:
# Stocare istorice per user_id
conversation_histories = {}
MAX_HISTORY = 10  # Numărul maxim de mesaje păstrate în context


def get_history(user_id: str) -> list:
    """Returnează istoricul conversației pentru un user."""
    return conversation_histories.get(user_id, [])


def update_history(user_id: str, role: str, content: str):
    """
    Adaugă un mesaj nou în istoricul conversației.
    Păstrează maxim MAX_HISTORY mesaje pentru a evita depășirea contextului.
    """
    if user_id not in conversation_histories:
        conversation_histories[user_id] = []

    conversation_histories[user_id].append({"role": role, "content": content})

    # Trunchiază istoricul dacă depășește limita
    if len(conversation_histories[user_id]) > MAX_HISTORY:
        conversation_histories[user_id] = conversation_histories[user_id][-MAX_HISTORY:]


def clear_history(user_id: str):
    """Resetează conversația pentru un user (ex: după farewell)."""
    conversation_histories[user_id] = []


print("✅ Sistem de gestionare context inițializat")

## Pasul 6 — Apelul către Groq API

In [ ]:
def call_llm(system_prompt: str, messages: list) -> str:
    """
    Trimite cererea către Groq API și returnează răspunsul generat.
    Input:
        system_prompt (str) — instrucțiunile pentru LLM
        messages (list)     — istoricul conversației
    Output:
        string — răspunsul generat de model
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            *messages
        ],
        temperature=0.7,      # Creativitate moderată
        max_tokens=300,       # Răspunsuri scurte
    )
    return response.choices[0].message.content


print("✅ Funcție apel LLM definită")

## Pasul 7 — Post-procesarea Răspunsului

Verificăm că răspunsul generat este sigur, relevant și nu depășește lungimea permisă.

In [ ]:
# Cuvinte interzise — extinde lista după necesități
FORBIDDEN_WORDS = []

# Răspuns fallback dacă ceva nu e ok
FALLBACK_RESPONSE = "Îmi pare rău, nu am putut procesa cererea ta. Te rog contactează suportul nostru direct."


def post_process(response: str) -> dict:
    """
    Validează și curăță răspunsul generat de LLM.
    Input:  response (str) — răspunsul brut de la LLM
    Output: dict cu 'text' (răspunsul final) și 'was_filtered' (bool)
    """
    was_filtered = False

    # Verificare răspuns gol
    if not response or not response.strip():
        return {"text": FALLBACK_RESPONSE, "was_filtered": True}

    # Verificare cuvinte interzise
    response_lower = response.lower()
    for word in FORBIDDEN_WORDS:
        if word in response_lower:
            return {"text": FALLBACK_RESPONSE, "was_filtered": True}

    # Trunchiază dacă e prea lung
    if len(response) > MAX_RESPONSE_LENGTH:
        response = response[:MAX_RESPONSE_LENGTH].rsplit(" ", 1)[0] + "..."
        was_filtered = True

    return {"text": response.strip(), "was_filtered": was_filtered}


print("✅ Funcție post-procesare definită")

## Pasul 8 — Logging Interacțiuni

Logăm fiecare interacțiune pentru analiză ulterioară și îmbunătățirea sistemului.

In [ ]:
import pandas as pd
from datetime import datetime

interaction_log = []


def log_interaction(user_id: str, user_message: str, intent_data: dict,
                    response: str, was_filtered: bool):
    """
    Salvează o interacțiune în log pentru analiză ulterioară.
    """
    interaction_log.append({
        "timestamp":    datetime.now().isoformat(),
        "user_id":      user_id,
        "user_message": user_message,
        "intent":       intent_data.get("intent"),
        "confidence":   intent_data.get("confidence"),
        "response":     response,
        "was_filtered": was_filtered
    })


def export_log(filename: str = "interaction_log.csv"):
    """Exportă log-ul într-un fișier CSV."""
    df = pd.DataFrame(interaction_log)
    df.to_csv(filename, index=False)
    print(f"✅ Log exportat în {filename} — {len(df)} interacțiuni")
    return df


print("✅ Sistem de logging inițializat")

## Pasul 9 — Funcția Principală `generate_response()`

Aceasta este funcția finală pe care o expune Sarcina 2 către Sarcina 3.  
Sarcina 3 apelează doar această funcție — nu știe nimic despre LLM sau intenții.

In [ ]:
def generate_response(user_id: str, user_message: str, intent_data: dict = None) -> dict:
    """
    Funcția principală a Sarcinii 2.

    Input:
        user_id      (str)  — ID-ul utilizatorului (pentru gestionarea contextului)
        user_message (str)  — mesajul scris de utilizator
        intent_data  (dict) — output-ul de la Sarcina 1 (opțional, folosește mock dacă lipsește)

    Output:
        dict:
            'response'     (str)  — răspunsul generat
            'intent'       (str)  — intentul detectat
            'was_filtered' (bool) — dacă răspunsul a fost modificat de post-procesare
    """

    # 1. Dacă nu primim intent de la Sarcina 1, folosim mock-ul
    if intent_data is None:
        intent_data = get_intent_mock(user_message)

    # 2. Construim system prompt-ul pe baza intentului
    system_prompt = build_system_prompt(intent_data)

    # 3. Adăugăm mesajul utilizatorului în istoricul conversației
    update_history(user_id, "user", user_message)

    # 4. Apelăm LLM-ul cu tot contextul
    raw_response = call_llm(system_prompt, get_history(user_id))

    # 5. Post-procesăm răspunsul
    result = post_process(raw_response)

    # 6. Salvăm răspunsul în istoricul conversației
    update_history(user_id, "assistant", result["text"])

    # 7. Resetăm contextul dacă utilizatorul și-a luat rămas bun
    if intent_data.get("intent") == "farewell":
        clear_history(user_id)

    # 8. Logăm interacțiunea
    log_interaction(user_id, user_message, intent_data, result["text"], result["was_filtered"])

    return {
        "response":     result["text"],
        "intent":       intent_data.get("intent"),
        "was_filtered": result["was_filtered"]
    }


print("✅ Funcția generate_response() definită — Sarcina 2 gata")

## Pasul 10 — Testare & Demo

Testăm întregul flux cu o conversație simulată.

In [ ]:
# Simulăm o conversație completă cu un utilizator
test_messages = [
    "Bună ziua!",
    "Am o problemă cu contul meu, nu mă pot loga",
    "Am primit eroarea 403",
    "Mulțumesc, a funcționat!",
    "Pa!"
]

print("=" * 60)
print("DEMO CONVERSAȚIE")
print("=" * 60)

for msg in test_messages:
    print(f"\n👤 User: {msg}")
    result = generate_response(user_id="user_001", user_message=msg)
    print(f"🤖 Bot [{result['intent']}]: {result['response']}")
    if result['was_filtered']:
        print("   ⚠️  Răspuns filtrat/trunchiat")

print("\n" + "=" * 60)

In [ ]:
# Exportăm log-ul conversației
df_log = export_log("interaction_log.csv")
df_log

## Rezumat — Ce expune Sarcina 2 către Sarcina 3

### Input așteptat:
```python
{
    "user_id": "user_001",
    "message": "Nu îmi merge aplicația",
    "intent_data": {                    # opțional — vine de la Sarcina 1
        "intent": "complaint",
        "confidence": 0.87,
        "entities": {"issue": "aplicație"}
    }
}
```

### Output returnat:
```python
{
    "response": "Îmi pare rău să aud asta! Poți să îmi dai mai multe detalii despre eroarea întâlnită?",
    "intent": "complaint",
    "was_filtered": False
}
```

### Funcții disponibile:
| Funcție | Rol |
|---|---|
| `generate_response(user_id, message, intent_data)` | Funcția principală |
| `clear_history(user_id)` | Resetează contextul unui user |
| `export_log()` | Exportă log-ul interacțiunilor |
| `get_intent_mock(text)` | Mock Sarcina 1 — înlocuit când e gata |